In [47]:
from pathlib import Path

import pandas as pd


In [48]:
input_path = Path("../data/csv/Hackaton/Detalle_entrega.csv")
output_path = Path("../data/custom/orders.csv")

In [49]:
rename_columns = {
    "FECHA": "date",
    "Transporte": "transport_id",
    "Destinatario mcía.": "location_id",
    "Nombre 1": "client",
    "Material": "item",
    "Cantidad entrega": "amount",
    "Denominación": "item_description",
    "calle": "street",
    "Un.medida venta": "unit",
}

keep_columns = [
    "date",
    "transport_id",
    "location_id",
    "client",
    "item",
    "item_description",
    "amount",
    "unit",
    "street",
]

df = pd.read_csv(input_path)
df = df.rename(columns=rename_columns)

# Keep only expected columns that actually exist in source.
available_keep_columns = [c for c in keep_columns if c in df.columns]
df = df[available_keep_columns].copy()

df.head()

,date,transport_id,location_id,client,item,item_description,amount,unit
0,30/01/2026,11420136,JACINT MAS CORNET,LOS TERESITOS,0CF0357,BONKA ESSENTIA ESPRESSO ITALIANO 1KG,2,UN
1,30/01/2026,11420136,JACINT MAS CORNET,LOS TERESITOS,0AM1290,XPLICIT AZUCAR 7GR 1000U,1,CAJ
2,30/01/2026,11420136,JACINT MAS CORNET,LOS TERESITOS,0AM1634,XPLICIT EDULCORANTE 1G 150U,1,ZPR
3,30/01/2026,11420334,JORGE ESCALANTE GUARERAY,HOSPITAL DE MANLLEU CUINA,0AG0007,"FONT D.OR NATURAL 1,5L PET 12U",38,CAJ
4,30/01/2026,11420334,JORGE ESCALANTE GUARERAY,HOSPITAL DE MANLLEU CAFETERIA,0ZU0027,LAMBDA MELOCOTON SIN 20CL 24U,2,CAJ


In [ ]:
unit_factors = {
    "un": 1,
    "caj": 9,
    "pak": 18,
    "bot": 1,
    "brl": 36,
}

df["unit"] = df["unit"].astype(str).str.strip().str.lower()

df["amount"] = pd.to_numeric(df["amount"], errors="coerce")
df = df[df["amount"].notna()].copy()

# Normalize key string fields used by downstream clustering.
for col in ["date", "transport_id", "item", "item_description", "client"]:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

unit_multiplier = df["unit"].map(unit_factors)
df["converted_amount"] = df["amount"] * unit_multiplier

unknown_units = sorted(df.loc[unit_multiplier.isna(), "unit"].dropna().unique())
if unknown_units:
    print(
        "Units without conversion factor preserved in output:",
        ", ".join(unknown_units),
    )

df.head(20)

,date,transport_id,location_id,client,item,item_description,amount,unit,converted_amount
0,30/01/2026,11420136,JACINT MAS CORNET,LOS TERESITOS,0CF0357,BONKA ESSENTIA ESPRESSO ITALIANO 1KG,2,un,2
1,30/01/2026,11420136,JACINT MAS CORNET,LOS TERESITOS,0AM1290,XPLICIT AZUCAR 7GR 1000U,1,caj,9
3,30/01/2026,11420334,JORGE ESCALANTE GUARERAY,HOSPITAL DE MANLLEU CUINA,0AG0007,"FONT D.OR NATURAL 1,5L PET 12U",38,caj,342
4,30/01/2026,11420334,JORGE ESCALANTE GUARERAY,HOSPITAL DE MANLLEU CAFETERIA,0ZU0027,LAMBDA MELOCOTON SIN 20CL 24U,2,caj,18
5,30/01/2026,11420334,JORGE ESCALANTE GUARERAY,HOSPITAL DE MANLLEU CAFETERIA,0ZU0028,LAMBDA NARANJA SIN 20CL 24U,2,caj,18
6,30/01/2026,11420334,JORGE ESCALANTE GUARERAY,BAR OLIVEDA,ED13,ESTRELLA DAMM 1/3 RET. PP,2,caj,18
7,30/01/2026,11420334,JORGE ESCALANTE GUARERAY,BAR OLIVEDA,CJ13,CAJA DAMM+BOT.1/3RET VACIO EN P.PLAS.A13,2,caj,18
8,30/01/2026,11420334,JORGE ESCALANTE GUARERAY,BAR OLIVEDA,0AG0183,VICHY CATALAN GAS 30CL RET 24U,2,caj,18
9,30/01/2026,11420334,JORGE ESCALANTE GUARERAY,BAR OLIVEDA,3ENV0236,C.C. AGUA 1/3 VICHY-FONT D'OR,2,caj,18
10,30/01/2026,11420334,JORGE ESCALANTE GUARERAY,BAR OLIVEDA,0RF0014,ZALA GASEOSA 50CL VID. RET 20U,2,caj,18


In [51]:
df.to_csv(output_path, index=False)